<a href="https://colab.research.google.com/github/paurnavi25/Mini-RAG-Question-Answerer/blob/main/Mini_MultiPDF_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
os.listdir()

['.config',
 'BCS515B-module-2-pdf.pdf',
 'BCS515B-module-3-pdf.pdf',
 'BCS515B-module-1-pdf.pdf',
 'sample_data']

In [ ]:
!pip install -q pypdf sentence-transformers faiss-cpu


In [ ]:
from pypdf import PdfReader
import os

documents = {}

for filename in os.listdir():
    if filename.endswith(".pdf"):
        reader = PdfReader(filename)

        text = ""
        for page in reader.pages:
            page_text = page.extract_text()
            if page_text:
                text += page_text + "\n"

        documents[filename] = text

print("PDFs loaded:", list(documents.keys()))

PDFs loaded: ['BCS515B-module-2-pdf.pdf', 'BCS515B-module-3-pdf.pdf', 'BCS515B-module-1-pdf.pdf']


In [ ]:
for filename, text in documents.items():
    print(filename, "→", len(text), "characters")

BCS515B-module-2-pdf.pdf → 30630 characters
BCS515B-module-3-pdf.pdf → 17859 characters
BCS515B-module-1-pdf.pdf → 43191 characters


In [ ]:
for filename, text in documents.items():
    print("\n" + "="*60)
    print(filename)
    print("="*60)
    print(text[:500])


BCS515B-module-2-pdf.pdf
k 
1 
  
 
 
 
Artificial Intelligence   
 
Module 2 
PROBLEM-SOLVING 
 
 
 
1.  PROBLEM-SOLVING AGENTS 
Goal formulation, based on the current situation and the agent's performance measure, is the first step in 
problem solving. 
Problem formulation is the process of deciding what actions and states to consider, given a goal. 
An agent with several immediate options of unknown value can decide what to do by first examining future 
actions that eventually lead to states of known value. 
 
The pr

BCS515B-module-3-pdf.pdf
Module 3 
3.5 INFORMED SEARCH STRATEGIES 
Informed search strategy is one that uses problem -specific knowledge beyond the definition of the 
problem itself—can find solutions more efficiently than can an uninformed strategy. The general approach 
we consider is called best-first search. Best-first search is an instance of the general algorithm in which 
a node is  selected for expansion based on an evaluation function , f(n). The evaluation f

In [ ]:
def create_chunks(text, chunk_size=500, overlap=100):
    chunks = []

    start = 0

    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end]

        chunks.append(chunk)

        start += chunk_size - overlap

    return chunks

In [ ]:
all_chunks = []

for filename, text in documents.items():
    chunks = create_chunks(text)

    for i, chunk in enumerate(chunks):
        all_chunks.append({
            "text": chunk,
            "source": filename,
            "chunk_id": i
        })

print("Total chunks:", len(all_chunks))

Total chunks: 230


In [ ]:
for i in range(3):
    print("\n--- CHUNK", i, "---")
    print("Source:", all_chunks[i]["source"])
    print(all_chunks[i]["text"][:500])


--- CHUNK 0 ---
Source: BCS515B-module-2-pdf.pdf
k 
1 
  
 
 
 
Artificial Intelligence   
 
Module 2 
PROBLEM-SOLVING 
 
 
 
1.  PROBLEM-SOLVING AGENTS 
Goal formulation, based on the current situation and the agent's performance measure, is the first step in 
problem solving. 
Problem formulation is the process of deciding what actions and states to consider, given a goal. 
An agent with several immediate options of unknown value can decide what to do by first examining future 
actions that eventually lead to states of known value. 
 
The pr

--- CHUNK 1 ---
Source: BCS515B-module-2-pdf.pdf
at to do by first examining future 
actions that eventually lead to states of known value. 
 
The process of looking for a sequence of actions that reaches the goal is called search. A search algorithm 
takes a problem as input and returns a solution in the form of an action sequence.  Once a solution  is 
found, the actions it recommends can be carried out. This is called the execution phase. Th

In [ ]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [ ]:
chunk_texts = [chunk["text"] for chunk in all_chunks]

In [ ]:
embeddings = embedding_model.encode(
    chunk_texts,
    show_progress_bar=True
)

Batches:   0%|          | 0/8 [00:00<?, ?it/s]

In [ ]:
print(embeddings.shape)

(230, 384)


In [ ]:
import faiss
import numpy as np

embeddings = np.array(embeddings).astype("float32")

dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(embeddings)

print("Vectors stored:", index.ntotal)

Vectors stored: 230


In [ ]:
question = "What is AI?"

In [ ]:
question_embedding = embedding_model.encode(
    [question]
)

question_embedding = np.array(question_embedding).astype("float32")

In [ ]:
k = 3

distances, indices = index.search(
    question_embedding,
    k
)

In [ ]:
for i, idx in enumerate(indices[0]):
    print("\n" + "="*60)
    print("Result:", i + 1)
    print("Source:", all_chunks[idx]["source"])
    print("Distance:", distances[0][i])
    print(all_chunks[idx]["text"][:1000])


Result: 1
Source: BCS515B-module-1-pdf.pdf
Distance: 0.36644077
Artificial Intelligence-BCS515B 
 
 
 
Module 1 
 
 Artificial intelligence (AI) is a term to describe a branch of computer  science that is 
dedicated to creating intelligent machines that would learn to work and react like 
humans. 
 AI which stands for artificial intelligence refers to systems or machines that mimic 
human intelligence to perform tasks and can iteratively improve themselves based on 
the information they collect. 
 
1.1 WHAT IS AI? 
 
 The field of artificial intelligenc e

Result: 2
Source: BCS515B-module-1-pdf.pdf
Distance: 0.48649105
ased on 
the information they collect. 
 
1.1 WHAT IS AI? 
 
 The field of artificial intelligenc e, or AI, it attempts not just to understand but also to build 
intelligent entities. 
 AI currently encompasses a huge variety of subfields, ranging from the general (learning and 
perception) to the specific, such as playing chess, proving mathematical theorems, writ

In [ ]:
!pip install -q transformers accelerate

In [ ]:
from transformers import pipeline

generator = pipeline(
    "text-generation",
    model="google/flan-t5-base"
)

config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  990MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


...

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

[transformers] The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'AXK1ForCausalLM', 'AXK2ForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'Cohere2MoeForCausalLM', 'CohereCompassForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV2ForCausalLM', 'DeepseekV3ForCausalLM', 'DeepseekV32ForCausalLM', 'DeepseekV4ForCausalLM', 'DiffLlamaForCausalLM', 'DogeForCausalLM', 'Dots1ForCausalLM', 'ElectraForCausalLM', 'Emu3ForCausalLM', 'ErnieForCausalLM',

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "google/flan-t5-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

print("Model loaded successfully!")

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Model loaded successfully!


In [41]:
question = "What is Artificial intelligence and machine learning?"

inputs = tokenizer(question, return_tensors="pt")

outputs = model.generate(
    **inputs,
    max_new_tokens=200
)

answer = tokenizer.decode(outputs[0], skip_special_tokens=True)

print(answer)

machine learning


In [42]:
retrieved_context = ""
for idx in indices[0]:
    retrieved_context += all_chunks[idx]["text"] + "\n\n"

prompt = f"""
Answer the question using only the following context.

Context:
{retrieved_context}

Question:
{question}

Answer:
"""

inputs = tokenizer(prompt, return_tensors="pt", truncation=True)
outputs = model.generate(
    **inputs,
    max_new_tokens=200
)

answer = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(answer)

a branch of computer science that is dedicated to creating intelligent machines that would learn to work and react like humans
